# 01a — MAESTRO Western Classical: Exploratory Data Analysis

This notebook characterises the processed MAESTRO subset (150 files) across five
objective dimensions:

1. Duration distribution
2. Note density (notes/sec)
3. Pitch range
4. Pitch Class (PC) Entropy — primary metric from Yang & Lerch (2020)
5. Mean pitch class histogram

Results are saved to `results/` for use in the cross-tradition comparison (Step 7).


In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

_here = Path().resolve()
PROJECT_ROOT = _here
for _ in range(5):
    if (PROJECT_ROOT / "PROGRESS.md").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.midi_utils import (
    load_midi, analyse_midi, get_pitch_class_histogram,
    pitch_class_entropy, piano_roll_plot
)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

PC_LABELS = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mohammadashraf/Desktop/Thesis-Best


In [2]:
def build_stats_df(midi_dir, meta_df=None, id_col=None):
    """Analyse all MIDI files in midi_dir; optionally merge metadata."""
    midi_files = sorted(midi_dir.glob("*.mid")) + sorted(midi_dir.glob("*.midi"))
    print(f"Analysing {len(midi_files)} MIDI files ...")
    records = [analyse_midi(p) for p in midi_files]
    df = pd.DataFrame(records)
    if "error" in df.columns:
        bad = df["error"].notna().sum()
        if bad:
            print(f"  Warning: {bad} files failed to load.")
        df = df[df["error"].isna()].drop(columns=["error"])
    df["filename"] = [Path(p).name for p in df["path"]]
    if meta_df is not None and id_col is not None:
        df = df.merge(meta_df, left_on="filename", right_on=id_col, how="left")
    return df

In [3]:
def summary_panel(df, tradition_name, save_path):
    """4-panel summary figure: duration, note density, pitch range, PC entropy."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(f"{tradition_name} — EDA Summary (n={len(df)})", fontsize=13, y=1.01)

    # Duration
    ax = axes[0, 0]
    df["duration_s"].div(60).plot.hist(bins=25, ax=ax, color="steelblue", edgecolor="white")
    ax.axvline(df["duration_s"].mean()/60, color="red", linestyle="--", label=f"mean={df['duration_s'].mean()/60:.1f} min")
    ax.set_xlabel("Duration (minutes)")
    ax.set_title("Duration Distribution")
    ax.legend(fontsize=8)

    # Note density
    ax = axes[0, 1]
    df["note_density"].plot.hist(bins=25, ax=ax, color="seagreen", edgecolor="white")
    ax.axvline(df["note_density"].mean(), color="red", linestyle="--", label=f"mean={df['note_density'].mean():.2f}")
    ax.set_xlabel("Notes per second")
    ax.set_title("Note Density Distribution")
    ax.legend(fontsize=8)

    # Pitch range
    ax = axes[1, 0]
    df["pitch_range"].plot.hist(bins=25, ax=ax, color="darkorange", edgecolor="white")
    ax.axvline(df["pitch_range"].mean(), color="red", linestyle="--", label=f"mean={df['pitch_range'].mean():.1f}")
    ax.set_xlabel("Pitch range (semitones)")
    ax.set_title("Pitch Range Distribution")
    ax.legend(fontsize=8)

    # PC entropy
    ax = axes[1, 1]
    df["pc_entropy"].plot.hist(bins=25, ax=ax, color="mediumpurple", edgecolor="white")
    ax.axvline(df["pc_entropy"].mean(), color="red", linestyle="--", label=f"mean={df['pc_entropy'].mean():.3f}")
    ax.set_xlabel("Pitch Class Entropy (bits)")
    ax.set_title("PC Entropy Distribution\n(Yang & Lerch, 2020)")
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path.relative_to(PROJECT_ROOT)}")


def mean_pc_histogram_plot(df_midi_paths, tradition_name, save_path):
    """Plot the mean pitch class histogram across all pieces."""
    hists = []
    for p in df_midi_paths:
        pm = load_midi(p)
        if pm:
            hists.append(get_pitch_class_histogram(pm))
    if not hists:
        return
    mean_hist = np.mean(hists, axis=0)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(PC_LABELS, mean_hist, color="steelblue", edgecolor="white")
    ax.set_xlabel("Pitch class")
    ax.set_ylabel("Mean relative frequency")
    ax.set_title(f"{tradition_name} — Mean Pitch Class Histogram (n={len(hists)})")
    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path.relative_to(PROJECT_ROOT)}")

In [4]:
MIDI_DIR = PROJECT_ROOT / "data" / "processed" / "western_classical" / "midi"
META_CSV = PROJECT_ROOT / "data" / "metadata" / "maestro_selected.csv"

if not MIDI_DIR.exists() or not any(MIDI_DIR.iterdir()):
    raise FileNotFoundError(
        "No MIDI files found. Run notebook 00a_maestro_prep.ipynb first."
    )

meta = pd.read_csv(META_CSV)
print(f"Metadata rows: {len(meta)}")
meta.head(3)

Metadata rows: 150


,canonical_composer,canonical_title,split,year,midi_filename,audio_filename,duration,processed_filename
0,Franz Schubert,"Sonata in D Major, D850",train,2004,2004/MIDI-Unprocessed_XP_15_R2_2004_01_ORIG_MI...,2004/MIDI-Unprocessed_XP_15_R2_2004_01_ORIG_MI...,351.639988,2004_MIDI-Unprocessed_XP_15_R2_2004_01_ORIG_MI...
1,Johann Sebastian Bach / Ferruccio Busoni,"Chaconne in D Minor, BWV 1004",train,2004,2004/MIDI-Unprocessed_SMF_13_01_2004_01-05_ORI...,2004/MIDI-Unprocessed_SMF_13_01_2004_01-05_ORI...,832.292417,2004_MIDI-Unprocessed_SMF_13_01_2004_01-05_ORI...
2,Johann Sebastian Bach,French Suite No. 5 in G Major,train,2004,2004/MIDI-Unprocessed_SMF_22_R1_2004_01-04_ORI...,2004/MIDI-Unprocessed_SMF_22_R1_2004_01-04_ORI...,792.520098,2004_MIDI-Unprocessed_SMF_22_R1_2004_01-04_ORI...


## 1. Load and compute statistics

In [5]:
stats = build_stats_df(MIDI_DIR)
print(f"Files analysed: {len(stats)}")
print("\nDescriptive statistics:")
print(stats[["duration_s", "note_count", "note_density", "pitch_range", "pc_entropy"]].describe().round(3))

Analysing 150 MIDI files ...


Files analysed: 150

Descriptive statistics:
       duration_s  note_count  note_density  pitch_range  pc_entropy
count     150.000     150.000       150.000      150.000     150.000
mean      594.812    5877.347        10.787       68.173       3.356
std       485.008    4355.336         3.586       11.154       0.138
min        70.710     877.000         3.855       45.000       2.830
25%       280.143    2780.750         8.082       60.000       3.255
50%       484.250    4708.000        10.174       72.000       3.372
75%       720.952    7230.000        12.541       76.750       3.452
max      2423.500   21420.000        20.988       86.000       3.578


## 2. Distribution plots

In [6]:
summary_panel(stats, "Western Classical (MAESTRO)", RESULTS_DIR / "eda_maestro_summary.png")

Saved → results/eda_maestro_summary.png


## 3. Mean pitch class histogram

In [7]:
mean_pc_histogram_plot(stats["path"].tolist(), "Western Classical (MAESTRO)",
                       RESULTS_DIR / "eda_maestro_pc_histogram.png")

Saved → results/eda_maestro_pc_histogram.png


## 4. Composer and year breakdown

In [8]:
# Merge with metadata for labelled analysis
stats_meta = stats.copy()
stats_meta["processed_filename"] = stats_meta["filename"]
stats_meta = stats_meta.merge(
    meta[["processed_filename", "canonical_composer", "year", "duration"]],
    on="processed_filename", how="left"
)

print("Top 10 composers by file count:")
print(stats_meta["canonical_composer"].value_counts().head(10).to_string())
print("\nMean PC entropy by year:")
print(stats_meta.groupby("year")["pc_entropy"].mean().round(4).to_string())

Top 10 composers by file count:
canonical_composer
Franz Schubert             29
Frédéric Chopin            23
Ludwig van Beethoven       18
Johann Sebastian Bach      17
Franz Liszt                12
Alexander Scriabin          6
Joseph Haydn                5
Domenico Scarlatti          5
Claude Debussy              4
Wolfgang Amadeus Mozart     4

Mean PC entropy by year:
year
2004    3.3499
2006    3.3918
2008    3.3781
2009    3.3713
2011    3.3242
2013    3.3207
2014    3.3775
2015    3.3344
2017    3.3461
2018    3.3907


## 5. Sample piano roll

In [9]:
sample_path = stats["path"].iloc[0]
pm = load_midi(sample_path)
sample_name = Path(sample_path).stem[:60]

fig, ax = plt.subplots(figsize=(14, 4))
piano_roll_plot(pm, ax, time_start=30, time_end=60,
                title=f"Piano roll sample — {sample_name}")
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "eda_maestro_piano_roll.png"), dpi=150)
plt.show()

## 6. Save summary statistics

In [10]:
stats["tradition"] = "western_classical"
stats.to_csv(RESULTS_DIR / "eda_maestro_stats.csv", index=False)
print("Saved → results/eda_maestro_stats.csv")

print("\n=== MAESTRO EDA Summary ===")
print(f"Files          : {len(stats)}")
print(f"Duration       : {stats['duration_s'].mean()/60:.1f} min mean  "
      f"({stats['duration_s'].min()/60:.1f} – {stats['duration_s'].max()/60:.1f})")
print(f"Note density   : {stats['note_density'].mean():.2f} notes/s mean")
print(f"Pitch range    : {stats['pitch_range'].mean():.1f} semitones mean")
print(f"PC entropy     : {stats['pc_entropy'].mean():.3f} bits mean")

Saved → results/eda_maestro_stats.csv

=== MAESTRO EDA Summary ===
Files          : 150
Duration       : 9.9 min mean  (1.2 – 40.4)
Note density   : 10.79 notes/s mean
Pitch range    : 68.2 semitones mean
PC entropy     : 3.356 bits mean
